In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))

import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, regularizers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from xgboost import XGBClassifier

import loading
import estimation as est

FACTOR_NAMES = est.FACTOR_NAMES
warnings.filterwarnings('ignore', category=ConvergenceWarning)  # unregularized LR on ~250-600 rows x 259 features warns often; expected, not a bug

In [ ]:
# Off-the-shelf models (paper Sec 3.2.3): unlike MT, each of these estimates a *separate*
# functional form per factor, with no shared structure across factors. Same walk-forward
# estimation procedure as the MT notebook (paper Sec 3.2.4) — expanding training window,
# 2-year validation window immediately before the test year, single held-out test year — so
# results are directly comparable to "Initial MT.ipynb".

data, feature_cols = est.build_labels_and_panel(
    loading.response_factors, loading.macro_predictors, loading.financial_predictors
)

print("data shape:", data.shape)
print("features:", len(feature_cols))
print("date range:", data.index.min().date(), "to", data.index.max().date())

In [ ]:
# --- Logistic Regression (paper's "LR") ---
# Plain, unregularized LR — the paper's Table IA1 lists a hyperparameter grid for the
# penalized version (EN) but not for LR, so no tuning is performed here either.

def lr_fit_predict(X_train, y_train, X_val, y_val, X_test):
    if len(np.unique(y_train)) < 2:  # degenerate fold guard: a factor with a one-sided sign run
        return np.full(len(X_test), y_train.mean())
    model = LogisticRegression(penalty=None, max_iter=5000)
    model.fit(X_train, y_train)
    return model.predict_proba(X_test)[:, 1]

lr_predictions = pd.DataFrame({
    f'{f}_prob': est.walk_forward_predict_single_task(data, feature_cols, f, lr_fit_predict)
    for f in FACTOR_NAMES
})
lr_predictions.to_csv('../results/lr_oos_predictions.csv')
print("LR done:", lr_predictions.shape)
lr_predictions.head()

In [ ]:
# --- Random Forest ---
# Fixed hyperparameters (500 trees, depth 5, sqrt(p) features per split) rather than the
# paper's per-fold grid search over Table IA1's {#Trees, Depth} grid, for tractability —
# same simplification as the MT notebook's fixed (l1, learning rate).

def rf_fit_predict(X_train, y_train, X_val, y_val, X_test):
    if len(np.unique(y_train)) < 2:
        return np.full(len(X_test), y_train.mean())
    model = RandomForestClassifier(
        n_estimators=500, max_depth=5, max_features='sqrt', random_state=0, n_jobs=-1
    )
    model.fit(X_train, y_train)
    return model.predict_proba(X_test)[:, 1]

rf_predictions = pd.DataFrame({
    f'{f}_prob': est.walk_forward_predict_single_task(data, feature_cols, f, rf_fit_predict)
    for f in FACTOR_NAMES
})
rf_predictions.to_csv('../results/rf_oos_predictions.csv')
print("RF done:", rf_predictions.shape)
rf_predictions.head()

In [ ]:
# --- XGBoost (stand-in for the paper's "GBT") ---
# Fixed hyperparameters (200 trees, depth 2, learning rate 0.1, 50% subsample — all valid
# points in Table IA1's GBT grid) rather than a per-fold grid search, for tractability.

def xgb_fit_predict(X_train, y_train, X_val, y_val, X_test):
    if len(np.unique(y_train)) < 2:
        return np.full(len(X_test), y_train.mean())
    model = XGBClassifier(
        n_estimators=200, max_depth=2, learning_rate=0.1, subsample=0.5,
        eval_metric='logloss', random_state=0,
    )
    model.fit(X_train, y_train)
    return model.predict_proba(X_test)[:, 1]

xgb_predictions = pd.DataFrame({
    f'{f}_prob': est.walk_forward_predict_single_task(data, feature_cols, f, xgb_fit_predict)
    for f in FACTOR_NAMES
})
xgb_predictions.to_csv('../results/xgb_oos_predictions.csv')
print("XGBoost done:", xgb_predictions.shape)
xgb_predictions.head()

In [ ]:
# --- LSTM (single-task, one model per factor) ---
# Architecture follows the paper's Internet Appendix IA3: one LSTM(32) layer, then 4 Dense(32)
# layers, then 2 Dense(8) layers, single sigmoid output — layer normalization (not batch norm),
# matching the paper's stated choice for dynamic/recurrent models. Same l1/lr/batch-size/epochs/
# patience as MT (single seed, fixed hyperparameters — same tractability simplification noted
# throughout this notebook).
#
# Design choice not specified in the paper text: a fixed 12-month lookback window per
# prediction (one macro/financial cycle) — see estimation.make_sequences.

LOOKBACK = 12

def build_lstm_model(n_features, l1_value=0.01, learning_rate=0.001, seed=0):
    tf.random.set_seed(seed)
    np.random.seed(seed)
    inputs = Input(shape=(LOOKBACK, n_features), name='predictor_history')
    x = layers.LSTM(32, kernel_regularizer=regularizers.l1(l1_value), name='lstm')(inputs)
    x = layers.LayerNormalization(name='lstm_ln')(x)
    for i in range(4):
        x = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l1(l1_value),
                          name=f'dense_{i+1}')(x)
        x = layers.LayerNormalization(name=f'ln_{i+1}')(x)
    for i in range(2):
        x = layers.Dense(8, activation='relu', kernel_regularizer=regularizers.l1(l1_value),
                          name=f'dense_small_{i+1}')(x)
    out = layers.Dense(1, activation='sigmoid', name='output')(x)
    model = Model(inputs, out, name='LSTM')
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='binary_crossentropy')
    return model

def lstm_fit_predict(X_train, y_train, X_val, y_val, X_test):
    if len(np.unique(y_train)) < 2:
        return np.full(len(X_test), y_train.mean())
    X_all = pd.concat([X_train, X_val, X_test])
    seqs_all = est.make_sequences(X_all, LOOKBACK)
    n_train, n_val = len(X_train), len(X_val)
    seq_train, seq_val, seq_test = seqs_all[:n_train], seqs_all[n_train:n_train + n_val], seqs_all[n_train + n_val:]

    model = build_lstm_model(n_features=X_train.shape[1])
    es = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    model.fit(seq_train, y_train, validation_data=(seq_val, y_val), epochs=200, batch_size=4,
              callbacks=[es], verbose=0)
    return model.predict(seq_test, verbose=0).flatten()

lstm_predictions = pd.DataFrame({
    f'{f}_prob': est.walk_forward_predict_single_task(data, feature_cols, f, lstm_fit_predict)
    for f in FACTOR_NAMES
})
lstm_predictions.to_csv('../results/lstm_oos_predictions.csv')
print("LSTM done:", lstm_predictions.shape)
lstm_predictions.head()

In [ ]:
# Benchmark: OOS classification accuracy (paper Table 1) and multi-factor timing Sharpe ratio
# (paper Table 3) for each off-the-shelf model, alongside the paper's published numbers and
# (if it has been run) this project's MT model for reference.

models = {'LR': lr_predictions, 'RF': rf_predictions, 'XGBoost (GBT)': xgb_predictions, 'LSTM': lstm_predictions}

try:
    mt_predictions = pd.read_csv('../results/mt_oos_predictions.csv', index_col=0, parse_dates=True)
    models['MT'] = mt_predictions
except FileNotFoundError:
    pass

rows = []
for name, pred in models.items():
    rows.append(est.benchmark_summary(pred, data, loading.response_factors, name))
summary = pd.DataFrame(rows).set_index('Model')

paper_reference = pd.DataFrame({
    'Mean Accuracy': {'LR': 53.1, 'RF': 55.6, 'XGBoost (GBT)': 54.8, 'LSTM': 53.8, 'MT': 55.4},
    'Sharpe Ratio': {'LR': 0.61, 'RF': 0.66, 'XGBoost (GBT)': 0.61, 'LSTM': 0.76, 'MT': 0.69},
}).T

print("This notebook, out-of-sample 1990-2021:")
print(summary.round(3))
print("\nPaper's published numbers, for reference (accuracy %, Sharpe ratio):")
print(paper_reference)